# RT-DETR-L: More Data Fine-Tuning Experiment

## Overview

This experiment investigates whether fine-tuning our best RT-DETR-L checkpoint on additional video frames improves object detection performance.

We started from the previously trained Fold 4 checkpoint and increased the number of training images while keeping the original validation split unchanged.

## Methodology

- **Architecture:** RT-DETR-L
- **Initial checkpoint:** Tuned Fold 4 `best.pt`
- **Training images:** 10,999 (previously 2,221)
- **Validation images:** 1,019
- **Training data:** Every second frame from eligible video sequences
- **Validation:** Original Fold 4 video-level holdout (12 sequences)
- **Additional training:** 4 epochs
- **Input resolution:** 960px
- **Batch size:** 1
- **Optimizer:** AdamW
- **Initial learning rate:** 0.00005
- **Random seed:** 42
- **Evaluation:** COCOeval, mAP@100

We excluded all validation video sequences from the additional training data to prevent video-level leakage.

Both the original checkpoint and the fine-tuned model were evaluated using the same validation images and evaluation procedure.

## Results

| Model | Validation mAP@100 |
|---|---:|
| Original RT-DETR-L Fold 4 | **0.286432** |
| RT-DETR-L Fold 4 — More Data | 0.258519 |
| Difference | -0.027913 |

Training required approximately 2.79 hours on an NVIDIA Tesla T4 GPU.

The additional fine-tuning did not improve validation performance. One possible explanation is the increased similarity between consecutive video frames, which may have introduced additional redundancy rather than sufficiently diverse training examples.

The original tuned checkpoints were therefore retained for our final ensemble.

## Reproducibility

This notebook requires the original VisDrone competition dataset, the prepared training dataset, and the Fold 4 checkpoint from `RT-DETR-L Tuned 5-Fold CV`.

The experiment uses `ultralytics==8.4.202`, PyTorch and `pycocotools`.

Generated outputs include:

- `preflight_report.json`
- `original_fold4_detections.json`
- `moredata_fold4_detections.json`
- `comparison.json`
- The fine-tuned `best.pt` checkpoint

Model weights and datasets are stored in Kaggle Inputs/Outputs and are not included in GitHub.

In [1]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/competitions/"
    "multiple-object-video-detection-hw-2-ucu-ai-2026"
)

PREP = Path(
    "/kaggle/input/datasets/vikhimiak/"
    "rtdetr-tuned-training-data/"
    "data_prepared/data_prepared"
)

assert (PREP / "fold4_val.txt").is_file()
assert (ROOT / "train/images").is_dir()
assert (ROOT / "val/images").is_dir()

# Original Fold 4 validation videos
val_names = [
    Path(line.strip()).name
    for line in (PREP / "fold4_val.txt").read_text().splitlines()
    if line.strip()
]

val_clips = {
    Path(name).stem.rsplit("_", 1)[0]
    for name in val_names
}

# All original labelled videos
folders = [
    folder
    for split in ("train", "val")
    for folder in (ROOT / split / "images").iterdir()
    if folder.is_dir()
]

available_clips = {folder.name for folder in folders}

print("ORIGINAL DATA CHECK")
print("=" * 60)

print("Original video clips:", len(available_clips))
print("Fold 4 validation clips:", len(val_clips))
print("Validation images:", len(val_names))

missing_clips = val_clips - available_clips
print("Missing validation clips:", missing_clips)

assert not missing_clips, (
    "Original clip names differ from prepared dataset!"
)

total = 0
eligible = 0
selected = 0

for folder in folders:
    images = sorted(folder.glob("*.jpg"))
    total += len(images)

    if folder.name in val_clips:
        continue

    eligible += len(images)
    selected += len(images[::2])

print("\nTotal labelled images:", total)
print("Training candidates excluding validation:", eligible)
print("Training images using every 2nd frame:", selected)
print("Previous Fold 4 training images: 2221")

print("\nValidation video leakage: NONE")
print("DATA CHECK PASSED")

ORIGINAL DATA CHECK
Original video clips: 63
Fold 4 validation clips: 12
Validation images: 1019
Missing validation clips: set()

Total labelled images: 27047
Training candidates excluding validation: 21972
Training images using every 2nd frame: 10999
Previous Fold 4 training images: 2221

Validation video leakage: NONE
DATA CHECK PASSED


In [2]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/competitions/"
    "multiple-object-video-detection-hw-2-ucu-ai-2026"
)

PREP = Path(
    "/kaggle/input/datasets/vikhimiak/"
    "rtdetr-tuned-training-data/data_prepared/data_prepared"
)

print("ORIGINAL ANNOTATIONS CHECK")
print("=" * 65)

for split in ("train", "val"):
    base = ROOT / split
    print(f"\n{split.upper()} STRUCTURE:")
    for p in sorted(base.iterdir()):
        print(" ", p.name, "(directory)" if p.is_dir() else "(file)")

# Inspect original Fold 4 training frames
val_names = [
    Path(s.strip()).name
    for s in (PREP / "fold4_val.txt").read_text().splitlines()
    if s.strip()
]

val_clips = {
    Path(name).stem.rsplit("_", 1)[0]
    for name in val_names
}

samples = []

for split in ("train", "val"):
    for clip in sorted((ROOT / split / "images").iterdir()):
        if not clip.is_dir() or clip.name in val_clips:
            continue

        images = sorted(clip.glob("*.jpg"))
        if images:
            samples.append(images[len(images) // 2])

print("\nEXAMPLE ANNOTATIONS")
print("=" * 65)

found = 0

for img in samples[:12]:
    split_root = img.parents[2]
    clip = img.parent.name

    candidates = [
        split_root / "labels" / clip / f"{img.stem}.txt",
        split_root / "annotations" / clip / f"{img.stem}.txt",
        split_root / "labels" / f"{clip}_{img.stem}.txt",
        split_root / "annotations" / f"{clip}_{img.stem}.txt",
        split_root / "annotations" / f"{clip}.txt",
    ]

    matches = [p for p in candidates if p.is_file()]

    print("\nIMAGE:", img.relative_to(ROOT))

    if matches:
        for p in matches:
            found += 1
            print("ANNOTATION:", p.relative_to(ROOT))
            lines = p.read_text(errors="replace").splitlines()
            print("Number of lines:", len(lines))
            print("First 2 lines:", lines[:2])
    else:
        print("No annotation found in common locations")

    prepared_label = (
        PREP / "labels" / f"{clip}_{img.stem}.txt"
    )

    if prepared_label.is_file():
        print(
            "PREPARED LABEL:",
            prepared_label.read_text().splitlines()[:2]
        )

print("\n" + "=" * 65)
print("Annotations located:", found)
print("CHECK COMPLETE — NO GPU TRAINING PERFORMED")

ORIGINAL ANNOTATIONS CHECK

TRAIN STRUCTURE:
  annotations_coco.json (file)
  data.yaml (file)
  ignore (directory)
  images (directory)
  labels (directory)

VAL STRUCTURE:
  annotations_coco.json (file)
  data.yaml (file)
  ignore (directory)
  images (directory)
  labels (directory)

EXAMPLE ANNOTATIONS

IMAGE: train/images/uav0000013_00000_v/0000135.jpg
ANNOTATION: train/labels/uav0000013_00000_v/0000135.txt
Number of lines: 31
First 2 lines: ['0 0.239955 0.833333 0.033482 0.153439', '0 0.239955 0.814153 0.036458 0.146825']

IMAGE: train/images/uav0000013_01392_v/0000060.jpg
ANNOTATION: train/labels/uav0000013_01392_v/0000060.txt
Number of lines: 29
First 2 lines: ['7 0.910342 0.736111 0.070685 0.152116', '0 0.912574 0.696429 0.057292 0.141534']

IMAGE: train/images/uav0000020_00406_v/0000251.jpg
ANNOTATION: train/labels/uav0000020_00406_v/0000251.txt
Number of lines: 19
First 2 lines: ['0 0.666295 0.914683 0.040923 0.165344', '0 0.709077 0.854497 0.043155 0.203704']
PREPARED LABEL

In [3]:

# ============================================================
# RT-DETR-L TUNED FOLD 4 — MORE DATA FINE-TUNING
# CPU preflight -> original COCOeval -> 4 epochs -> new COCOeval
#
# First run with RUN_TRAINING=False and Accelerator=None.
# Then change to True and use Save Version -> Save & Run All.
# ============================================================

RUN_TRAINING = True

from pathlib import Path
import json
import math
import time

COMP = Path(
    "/kaggle/input/competitions/"
    "multiple-object-video-detection-hw-2-ucu-ai-2026"
)

PREP = Path(
    "/kaggle/input/datasets/vikhimiak/"
    "rtdetr-tuned-training-data/"
    "data_prepared/data_prepared"
)

OLD_MODEL = Path(
    "/kaggle/input/notebooks/vikhimiak/"
    "rt-detr-l-tuned-5-fold-cv/"
    "rtdetr_tuned/cv_runs/tuned_cv_fold4/weights/best.pt"
)

OUT = Path("/kaggle/working/rtdetr_fold4_moredata")
OUT.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "person", "bicycle", "car", "van",
    "truck", "tricycle", "bus", "motor"
]

assert COMP.is_dir(), "Competition dataset missing"
assert PREP.is_dir(), "Prepared dataset missing"
assert OLD_MODEL.is_file(), "Fold 4 checkpoint missing"

print("=" * 65)
print("STEP 1 — FULL DATA PREFLIGHT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Reconstruct the original Fold 4 video holdout
# ------------------------------------------------------------

def read_names(path):
    return [
        Path(line.strip()).name
        for line in path.read_text().splitlines()
        if line.strip()
    ]


def video_id(name):
    return Path(name).stem.rsplit("_", 1)[0]


val_names = read_names(PREP / "fold4_val.txt")
val_clips = {video_id(name) for name in val_names}

assert len(val_names) == len(set(val_names)) == 1019
assert len(val_clips) == 12

clip_folders = [
    folder
    for split in ("train", "val")
    for folder in (COMP / split / "images").iterdir()
    if folder.is_dir()
]

assert len(clip_folders) == 63
assert len({p.name for p in clip_folders}) == 63
assert val_clips.issubset({p.name for p in clip_folders})

# ------------------------------------------------------------
# 2. Select every second frame, excluding held-out videos
# ------------------------------------------------------------

train_imgs = []
all_eligible = 0

for folder in sorted(clip_folders):

    if folder.name in val_clips:
        continue

    frames = sorted(folder.glob("*.jpg"))

    all_eligible += len(frames)
    train_imgs.extend(frames[::2])

val_imgs = [
    PREP / "images" / name
    for name in val_names
]

assert len(train_imgs) == len(set(train_imgs)) == 10999, (
    f"Unexpected training count: {len(train_imgs)}"
)

assert all_eligible == 21972
assert all(p.is_file() for p in val_imgs)
assert {
    p.parent.name for p in train_imgs
}.isdisjoint(val_clips), "Video leakage detected"

# ------------------------------------------------------------
# 3. Validate the COCO validation annotations
# ------------------------------------------------------------

with open(PREP / "coco/fold4_val.json") as f:
    coco_val = json.load(f)

assert len(coco_val["images"]) == 1019

assert {
    Path(img["file_name"]).name
    for img in coco_val["images"]
} == set(val_names)

categories = {
    int(c["id"]): c["name"]
    for c in coco_val["categories"]
}

assert categories == {
    i + 1: name for i, name in enumerate(CLASSES)
}, categories

# ------------------------------------------------------------
# 4. Check ALL training and validation YOLO labels
# ------------------------------------------------------------

def original_label_path(img):
    return (
        img.parents[2]
        / "labels"
        / img.parent.name
        / (img.stem + ".txt")
    )


def validate_label(path, image):
    assert path.is_file(), (
        f"Missing label: {path} for {image}"
    )

    count = 0

    for line_no, line in enumerate(
        path.read_text().splitlines(), 1
    ):
        if not line.strip():
            continue

        parts = line.split()

        assert len(parts) == 5, (
            f"Malformed label: {path}:{line_no}"
        )

        values = list(map(float, parts))

        assert all(math.isfinite(x) for x in values)
        assert values[0].is_integer()
        assert 0 <= int(values[0]) < 8

        assert all(
            0 <= x <= 1 for x in values[1:]
        ), f"Invalid normalized coordinates: {path}:{line_no}"

        assert values[3] > 0 and values[4] > 0

        count += 1

    return count


object_count = 0
matched_labels = 0

for img in train_imgs:

    original_label = original_label_path(img)

    object_count += validate_label(
        original_label, img
    )

    # Cross-check against previously prepared labels
    prepared_name = f"{img.parent.name}_{img.stem}.txt"
    prepared_label = PREP / "labels" / prepared_name

    if prepared_label.is_file():

        assert (
            original_label.read_text().strip()
            == prepared_label.read_text().strip()
        ), f"Original/prepared label mismatch: {img}"

        matched_labels += 1

for img in val_imgs:

    label = PREP / "labels" / f"{img.stem}.txt"

    object_count += validate_label(label, img)

# ------------------------------------------------------------
# 5. Create Ultralytics dataset configuration
# ------------------------------------------------------------

train_txt = OUT / "train_every2.txt"
val_txt = OUT / "val_original_fold4.txt"

train_txt.write_text(
    "\n".join(map(str, train_imgs)) + "\n"
)

val_txt.write_text(
    "\n".join(map(str, val_imgs)) + "\n"
)

config = {
    "path": str(OUT),
    "train": str(train_txt),
    "val": str(val_txt),
    "nc": 8,
    "names": CLASSES
}

data_yaml = OUT / "data.yaml"
data_yaml.write_text(json.dumps(config, indent=2))

report = {
    "training_images": len(train_imgs),
    "eligible_original_images": all_eligible,
    "validation_images": len(val_imgs),
    "held_out_clips": sorted(val_clips),
    "matched_prepared_labels": matched_labels,
    "total_objects_checked": object_count,
    "initial_checkpoint": str(OLD_MODEL)
}

(OUT / "preflight_report.json").write_text(
    json.dumps(report, indent=2)
)

print("Training images:", len(train_imgs))
print("Validation images:", len(val_imgs))
print("Validation clips:", len(val_clips))
print("Cross-checked labels:", matched_labels)
print("Validated objects:", object_count)
print("\nALL DATA PREFLIGHT CHECKS PASSED!")

# ============================================================
# TRAINING IS DISABLED UNLESS EXPLICITLY ENABLED
# ============================================================

if not RUN_TRAINING:

    print("\nNO TRAINING PERFORMED.")
    print("Set RUN_TRAINING=True only after preflight PASS.")

else:

    # --------------------------------------------------------
    # 6. Install and verify runtime
    # --------------------------------------------------------

    import sys
    import subprocess
    import importlib.metadata

    for package, version in [
        ("ultralytics", "8.4.202"),
        ("pycocotools", None)
    ]:
        try:
            installed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            installed = None

        if installed is None or (
            version is not None and installed != version
        ):
            target = (
                f"{package}=={version}"
                if version else package
            )
            subprocess.check_call([
                sys.executable, "-m", "pip",
                "install", "-q", target
            ])

    import numpy as np
    import torch

    from ultralytics import RTDETR
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    assert torch.cuda.is_available(), "GPU not enabled!"

    print("\nGPU:", torch.cuda.get_device_name(0))

    # --------------------------------------------------------
    # 7. Independent COCO evaluation
    # --------------------------------------------------------

    def evaluate_checkpoint(checkpoint, name):

        model = RTDETR(str(checkpoint))

        assert [
            model.names[i] for i in range(8)
        ] == CLASSES

        gt_path = PREP / "coco/fold4_val.json"
        coco_gt = COCO(str(gt_path))

        detections = []

        for idx, item in enumerate(coco_val["images"], 1):

            image_path = PREP / item["file_name"]

            assert image_path.is_file(), image_path

            result = model.predict(
                source=str(image_path),
                imgsz=960,
                conf=0.01,
                iou=0.7,
                max_det=500,
                device=0,
                verbose=False
            )[0]

            height, width = result.orig_shape

            assert (width, height) == (
                item["width"], item["height"]
            )

            for box in result.boxes:

                cls = int(box.cls.item())
                score = float(box.conf.item())

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                x1 = max(0.0, min(float(x1), width))
                x2 = max(0.0, min(float(x2), width))
                y1 = max(0.0, min(float(y1), height))
                y2 = max(0.0, min(float(y2), height))

                if x2 <= x1 or y2 <= y1:
                    continue

                detections.append({
                    "image_id": item["id"],
                    "category_id": cls + 1,
                    "bbox": [
                        x1, y1,
                        x2 - x1,
                        y2 - y1
                    ],
                    "score": score
                })

            if idx % 250 == 0:
                print(
                    f"{name}: {idx}/1019 images",
                    flush=True
                )

        (OUT / f"{name}_detections.json").write_text(
            json.dumps(detections)
        )

        coco_dt = coco_gt.loadRes(detections)

        evaluator = COCOeval(
            coco_gt, coco_dt, "bbox"
        )

        evaluator.params.imgIds = sorted(
            coco_gt.getImgIds()
        )

        evaluator.params.maxDets = [1, 100, 500]

        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

        p = evaluator.eval["precision"][:, :, :, 0, 1]
        valid = p[p > -1]

        ap = float(valid.mean())

        (OUT / f"{name}_coco_metrics.json").write_text(
            json.dumps({
                "mAP_maxDets100": ap,
                "images": 1019,
                "detections": len(detections)
            }, indent=2)
        )

        del model, coco_gt, coco_dt, evaluator
        torch.cuda.empty_cache()

        print(f"{name} COCO mAP@100: {ap:.6f}")

        return ap

    # --------------------------------------------------------
    # 8. Verify original Fold 4 BEFORE training
    # --------------------------------------------------------

    print("\nVERIFYING ORIGINAL FOLD 4...")

    old_ap = evaluate_checkpoint(
        OLD_MODEL,
        "original_fold4"
    )

    # Sanity check, not an exact reproducibility requirement
    assert 0.27 <= old_ap <= 0.31, (
        f"Unexpected original Fold 4 mAP: {old_ap:.6f}"
    )
    
    print(f"Original Fold 4 baseline for comparison: {old_ap:.6f}")
    
    if abs(old_ap - 0.297715) > 0.01:
        print(
            "WARNING: Original evaluation differs from the "
            "previous run. New and old models will be compared "
            "using the same current evaluation pipeline."
        )

    print("\nORIGINAL MODEL VERIFIED!")

    # --------------------------------------------------------
    # 9. Fine-tune RT-DETR-L on more training frames
    # --------------------------------------------------------

    print("\nSTARTING MORE-DATA FINE-TUNING...")

    start = time.perf_counter()

    model = RTDETR(str(OLD_MODEL))

    assert [
        model.names[i] for i in range(8)
    ] == CLASSES

    model.train(
        data=str(data_yaml),

        project=str(OUT / "cv_runs"),
        name="fold4_moredata_4ep",

        epochs=4,
        imgsz=960,
        batch=1,
        nbs=8,

        optimizer="AdamW",
        lr0=0.00005,
        lrf=0.1,

        warmup_epochs=0.5,
        warmup_bias_lr=0.0,
        scale=0.5,

        device=0,
        workers=2,
        seed=42,
        deterministic=False,

        amp=True,
        val=True,

        save=True,
        save_period=1,
        plots=False,
        cache=False,
        exist_ok=True
    )

    best = (
        OUT
        / "cv_runs"
        / "fold4_moredata_4ep"
        / "weights"
        / "best.pt"
    )

    assert best.is_file(), (
        f"New checkpoint not found: {best}"
    )

    print(
        "Training time:",
        round(
            (time.perf_counter() - start) / 3600,
            2
        ),
        "hours"
    )

    del model
    torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 10. Evaluate fine-tuned model on SAME validation fold
    # --------------------------------------------------------

    print("\nEVALUATING NEW MODEL...")

    new_ap = evaluate_checkpoint(
        best,
        "moredata_fold4"
    )

    comparison = {
        "original_fold4_mAP100": old_ap,
        "moredata_fold4_mAP100": new_ap,
        "difference": new_ap - old_ap,
        "improved": bool(new_ap > old_ap),
        "training_images": len(train_imgs),
        "validation_images": len(val_imgs),
        "new_checkpoint": str(best)
    }

    (OUT / "comparison.json").write_text(
        json.dumps(comparison, indent=2)
    )

    print("\n" + "=" * 65)
    print("MORE-DATA EXPERIMENT COMPLETE")
    print("=" * 65)

    print("Original Fold 4 mAP@100:", round(old_ap, 6))
    print("More-data Fold 4 mAP@100:", round(new_ap, 6))
    print("Improvement:", round(new_ap - old_ap, 6))
    print("New checkpoint:", best)

    print("\nALL RESULTS SAVED TO:", OUT)
    print("CHECK VALIDATION BEFORE KAGGLE SUBMISSION.")

STEP 1 — FULL DATA PREFLIGHT
Training images: 10999
Validation images: 1019
Validation clips: 12
Cross-checked labels: 2221
Validated objects: 528869

ALL DATA PREFLIGHT CHECKS PASSED!
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

GPU: Tesla T4

VERIFYING ORIGINAL FOLD 4...
loading annotations into memory...
Done (t=0.21s)
creating index...
index created!
original_fold4: 250/1019 images
original_fold4: 500/1019 images
original_fold4: 750/1019 images
original_fold4: 1000/1019 images
Loading and preparing results...
DONE (t=0.78s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=83.81s).
Accumulating evaluation results...
DONE (t=4.69s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   a

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/4      2.33G     0.5307     0.4426    0.05773         37        960: 100% ━━━━━━━━━━━━ 10999/10999 4.5it/s 40:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 510/510 10.0it/s 50.8s
                   all       1019      54870      0.581       0.51      0.475      0.257

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/4      2.33G     0.5063     0.4302    0.05354        111        960: 100% ━━━━━━━━━━━━ 10999/10999 4.5it/s 40:28
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 510/510 10.1it/s 50.6s
                   all       1019      54870      0.519      0.484      0.427      0.219

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        4/4      2.33G     0.4881     0.4208    0.05073         21        960: 100% ━━━━━━━━━━━━ 10999/10999 4.5it/s 40:24
                 Class     Images